# Notebook 08 — Sensitivity & Decision Analysis

## Objective
Determine which operational investments and capacity
decisions provide the greatest modeled economic value.

## Business Questions
1. What is the value of additional supplier capacity?
2. Does expanding manufacturing capacity reduce costs?
3. How sensitive is the transportation strategy to freight prices?
4. How much demand growth can the network support?
5. Which operational investments should be prioritized?

## Methodology
Use the validated Pyomo optimization model to perform
controlled parameter changes and reoptimize the network.

Compare each result against the original optimized
cost of $404,131.20.

## Outputs
- Capacity sensitivity tables
- Transportation cost sensitivity
- Demand growth feasibility analysis
- Marginal capacity value estimates
- Investment break-even thresholds
- Tableau-ready datasets
- Executive recommendations

## Important Assumptions
All experiments use hypothetical network parameters.
Capacity expansion costs are excluded from the
optimization objective and considered separately
when interpreting investment value.


In [1]:
# STEP 2: Initialize Notebook 08 — Sensitivity Analysis

import pandas as pd
import numpy as np
import pyomo.environ as pyo

from pathlib import Path
from IPython.display import display, Markdown

# --------------------------------------------------
# 1. Locate Project 2 repository
# --------------------------------------------------

current_path = Path.cwd().resolve()

repo_root = next(
    (
        p for p in [current_path, *current_path.parents]
        if (p / "data/assumptions/bom.csv").is_file()
    ),
    None
)

assert repo_root is not None, "Project 2 repository not found"

# --------------------------------------------------
# 2. Load required data
# --------------------------------------------------

file_paths = {
    "suppliers": "data/assumptions/suppliers.csv",
    "plants": "data/assumptions/plants.csv",
    "lanes": "data/assumptions/transport_lanes.csv",
    "demand": "data/processed/regional_demand_12_weeks.csv",
    "optimized_cost": "outputs/tables/optimized_cost_breakdown.csv",
    "scenario_comparison": "outputs/tables/scenario_comparison.csv",
    "scenario_audit": "outputs/tables/scenario_audit.csv",
    "scenario_operations": "outputs/tables/scenario_operational_response.csv"
}

sensitivity_data = {}

for name, relative_path in file_paths.items():

    path = repo_root / relative_path

    assert path.is_file(), f"Missing required file: {path}"

    sensitivity_data[name] = pd.read_csv(path)

# --------------------------------------------------
# 3. Establish reference optimized cost
# --------------------------------------------------

reference_cost = float(
    sensitivity_data["optimized_cost"]["Cost_USD"].sum()
)

reference_scenario = sensitivity_data[
    "scenario_comparison"
].loc[
    lambda df: df["Scenario"] == "Reference"
]

assert len(reference_scenario) == 1

scenario_reference_cost = float(
    reference_scenario["Optimized_Cost_USD"].iloc[0]
)

assert np.isclose(
    reference_cost,
    scenario_reference_cost,
    atol=0.01
), "Notebook 04 and 07 reference costs disagree"

assert np.isclose(
    reference_cost,
    404131.20,
    atol=0.01
)

# --------------------------------------------------
# 4. Verify prior scenario validation
# --------------------------------------------------

audit = sensitivity_data["scenario_audit"]

optimal_audits = audit.loc[
    audit["Solver_Status"] == "Optimal"
]

assert len(optimal_audits) >= 1

assert (
    optimal_audits["Independent_Audit"] == "PASSED"
).all()

# Confirm Notebook 07 is present for model reuse
notebook07_path = (
    repo_root / "notebooks/07_scenario_analysis.ipynb"
)

assert notebook07_path.is_file()

# --------------------------------------------------
# 5. Print initialization results
# --------------------------------------------------

print("NOTEBOOK 08 INITIALIZED SUCCESSFULLY")
print("=" * 50)

print("Datasets loaded:", len(sensitivity_data))
print("Reference optimized cost: ${:,.2f}".format(reference_cost))
print("Notebook 07 audited scenarios:", len(optimal_audits))

print("Supplier records:", len(sensitivity_data["suppliers"]))
print("Transportation lanes:", len(sensitivity_data["lanes"]))
print("Demand records:", len(sensitivity_data["demand"]))

print("\nREFERENCE COST RECONCILIATION: PASSED")
print("SCENARIO AUDIT VERIFICATION: PASSED")
print("STEP 2: PASSED")

display(
    sensitivity_data["scenario_comparison"].round(2)
)


NOTEBOOK 08 INITIALIZED SUCCESSFULLY
Datasets loaded: 8
Reference optimized cost: $404,131.20
Notebook 07 audited scenarios: 5
Supplier records: 6
Transportation lanes: 4
Demand records: 24

REFERENCE COST RECONCILIATION: PASSED
SCENARIO AUDIT VERIFICATION: PASSED
STEP 2: PASSED


,Scenario,Status,Total_Demand_Units,Optimized_Cost_USD,Incremental_Cost_USD,Cost_Change_Pct
0,Reference,Optimal,6180,404131.20,0.00,0.00
1,Supplier_Disruption,Optimal,6180,417631.20,13500.00,3.34
2,Demand_Surge,Optimal,6336,418967.70,14836.50,3.67
3,Manufacturing_Disruption,Optimal,6180,405741.20,1610.00,0.40
4,Freight_Inflation,Optimal,6180,413920.45,9789.25,2.42


In [2]:
# STEP 3: Reuse Notebook 07's validated optimization engine

import json

# --------------------------------------------------
# 1. Locate Notebook 07
# --------------------------------------------------

notebook07_path = (
    repo_root / "notebooks" / "07_scenario_analysis.ipynb"
)

assert notebook07_path.is_file()

# Read the saved notebook
with open(notebook07_path, "r", encoding="utf-8") as f:
    notebook07 = json.load(f)

# --------------------------------------------------
# 2. Extract the model setup and solver cells
# --------------------------------------------------

required_steps = [
    "# STEP 2:",
    "# STEP 3:",
    "# STEP 4A:",
    "# STEP 4B:",
    "# STEP 5A:"
]

def get_notebook_cell(step_prefix):

    matches = []

    for cell in notebook07["cells"]:

        if cell["cell_type"] != "code":
            continue

        source = cell["source"]

        if isinstance(source, list):
            source = "".join(source)

        if source.strip().startswith(step_prefix):
            matches.append(source)

    assert len(matches) == 1, (
        f"Expected exactly one Notebook 07 cell "
        f"starting with {step_prefix}; found {len(matches)}"
    )

    return matches[0]


# --------------------------------------------------
# 3. Execute only the reusable model setup
# --------------------------------------------------

# Separate namespace prevents overwriting
# Notebook 08's own variables.

engine = {"__name__": "notebook08_scenario_engine"}

for step in required_steps:

    source_code = get_notebook_cell(step)

    exec(
        compile(
            source_code,
            f"notebook07_{step}",
            "exec"
        ),
        engine
    )

    print("Loaded:", step)

# This executes only code from our own saved Notebook 07.
# It does not rerun the four disruption experiments.

# --------------------------------------------------
# 4. Verify the reusable functions exist
# --------------------------------------------------

assert callable(engine["build_scenario_model"])
assert callable(engine["solve_scenario"])

assert len(engine["supplier_capacity"]) == 6
assert len(engine["weeks"]) == 12

# --------------------------------------------------
# 5. Independently reproduce the reference solution
# --------------------------------------------------

reference_spec = {
    "Scenario": "Sensitivity_Reference",
    "Type": "reference"
}

reference_result_08 = engine["solve_scenario"](
    reference_spec
)

assert reference_result_08["Status"] == "Optimal", (
    reference_result_08["Message"]
)

reproduced_cost = reference_result_08["Total_Cost_USD"]

assert np.isclose(
    reproduced_cost,
    reference_cost,
    atol=0.01
), "Sensitivity reference does not match Notebook 04"

# --------------------------------------------------
# 6. Confirm initialization
# --------------------------------------------------

print("\nNOTEBOOK 08 OPTIMIZATION ENGINE READY")
print("=" * 50)
print("Solver: HiGHS")
print("Planning horizon: 12 weeks")
print("Supplier-component combinations: 6")

print(
    "Reproduced optimized cost: "
    f"${reproduced_cost:,.2f}"
)

print("Reference cost reconciliation: PASSED")
print("STEP 3: PASSED")


NOTEBOOK 07 INITIALIZED SUCCESSFULLY
Datasets loaded: 11
Planning weeks: 12
Reference optimized cost: $ 404131.2
Customer demand: 6180


,Dataset,Records
0,bom,3
1,suppliers,6
2,plants,1
3,dcs,2
4,lanes,4
5,inventory,6
6,demand,24
7,baseline_components,36
8,baseline_plant,12
9,baseline_dcs,24


Loaded: # STEP 2:
DISRUPTION SCENARIOS DEFINED


,Scenario,Scenario_Type,Description
0,Reference,reference,Original optimized network
1,Supplier_Disruption,supplier_outage,Supplier A accepts no new orders in Weeks 4-5
2,Demand_Surge,demand_surge,Regional demand increases 10% in Weeks 6-8
3,Manufacturing_Disruption,plant_reduction,Plant capacity decreases 20% in Weeks 4-7
4,Freight_Inflation,freight_increase,Standard freight cost increases 50%



Four disruption scenarios + reference
STEP 3: PASSED
Loaded: # STEP 3:
REUSABLE OPTIMIZATION PARAMETERS READY
Weeks: 12
Supplier-component combinations: 6
Original demand: 6180
STEP 4A: PASSED
Loaded: # STEP 4A:
REUSABLE OPTIMIZATION MODEL CREATED
Decision variables: 204
Active constraints: 247

STEP 4B: PASSED
Loaded: # STEP 4B:
SCENARIO SOLVER CREATED SUCCESSFULLY
Solver: HiGHS
STEP 5A: PASSED
Loaded: # STEP 5A:

NOTEBOOK 08 OPTIMIZATION ENGINE READY
Solver: HiGHS
Planning horizon: 12 weeks
Supplier-component combinations: 6
Reproduced optimized cost: $404,131.20
Reference cost reconciliation: PASSED
STEP 3: PASSED


In [3]:
# STEP 4: Create reusable sensitivity experiment engine

import ast

# --------------------------------------------------
# 1. Load Notebook 07's independent audit function
# --------------------------------------------------

audit_source = get_notebook_cell("# STEP 6B:")
audit_tree = ast.parse(audit_source)

audit_function = next(
    (
        node for node in audit_tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "audit_scenario"
    ),
    None
)

assert audit_function is not None, (
    "Could not find audit_scenario in Notebook 07"
)

audit_module = ast.Module(
    body=[audit_function],
    type_ignores=[]
)

ast.fix_missing_locations(audit_module)

exec(
    compile(audit_module, str(notebook07_path), "exec"),
    engine
)

assert callable(engine["audit_scenario"])

# --------------------------------------------------
# 2. Preserve original optimization parameters
# --------------------------------------------------

base_supplier_capacity = dict(
    engine["supplier_capacity"]
)

base_plant_capacity = int(
    engine["plant_capacity"]
)

base_lane_cost = dict(
    engine["lane_cost"]
)

base_demand = dict(
    engine["base_demand"]
)

base_lane_modes = dict(zip(
    engine["lane_df"]["Lane_ID"],
    engine["lane_df"]["Transport_Mode"]
))

planning_weeks = list(engine["weeks"])
planning_horizon = len(planning_weeks)

# --------------------------------------------------
# 3. Function: temporarily modify inputs and solve
# --------------------------------------------------

def run_sensitivity_experiment(
    experiment_name,
    parameter_changes=None
):

    if parameter_changes is None:
        parameter_changes = {}

    original_values = {}

    try:
        for parameter, new_value in parameter_changes.items():

            assert parameter in {
                "supplier_capacity",
                "plant_capacity",
                "lane_cost",
                "base_demand"
            }

            original_values[parameter] = engine[parameter]
            engine[parameter] = new_value

        result = engine["solve_scenario"]({
            "Scenario": experiment_name,
            "Type": "reference"
        })

        # Independently audit each optimal solution
        if result["Status"] == "Optimal":

            audit_status = engine["audit_scenario"](
                result
            )

            assert audit_status == "PASSED"
            result["Independent_Audit"] = "PASSED"

        else:
            result["Independent_Audit"] = "NOT_APPLICABLE"

        return result

    finally:
        # Always restore original network parameters
        for parameter, original_value in original_values.items():
            engine[parameter] = original_value


# --------------------------------------------------
# 4. Confirm baseline functionality
# --------------------------------------------------

sensitivity_reference = run_sensitivity_experiment(
    "Reference"
)

assert sensitivity_reference["Status"] == "Optimal"

assert np.isclose(
    sensitivity_reference["Total_Cost_USD"],
    reference_cost,
    atol=0.01
)

assert sensitivity_reference["Independent_Audit"] == "PASSED"

# Confirm original parameters were preserved
assert engine["supplier_capacity"] == base_supplier_capacity
assert engine["plant_capacity"] == base_plant_capacity
assert engine["lane_cost"] == base_lane_cost
assert engine["base_demand"] == base_demand

# Store scenario results for subsequent analysis
sensitivity_results = {
    "Reference": sensitivity_reference
}

experiment_metadata = [{
    "Experiment": "Reference",
    "Category": "Reference",
    "Lever": "Original Network",
    "Change_Pct": 0,
    "Extra_Weekly_Capacity": 0,
    "Additional_Customer_Demand_Units": 0
}]

print("SENSITIVITY EXPERIMENT ENGINE READY")
print("=" * 50)
print("Original parameters preserved: PASSED")
print("Independent inventory audit: PASSED")
print(f"Reference cost: ${reference_cost:,.2f}")
print("STEP 4: PASSED")


SENSITIVITY EXPERIMENT ENGINE READY
Original parameters preserved: PASSED
Independent inventory audit: PASSED
Reference cost: $404,131.20
STEP 4: PASSED


In [4]:
# STEP 5: Supplier capacity sensitivity analysis

supplier_experiments = []

for supplier_id in ["SUP_A", "SUP_B"]:

    for increase_pct in [10, 20, 30]:

        factor = 1 + increase_pct / 100

        modified_capacity = base_supplier_capacity.copy()

        # Increase all eligible component capacities
        # for the selected supplier.
        for key, original_capacity in base_supplier_capacity.items():

            if key[0] == supplier_id:

                modified_capacity[key] = int(
                    np.floor(original_capacity * factor)
                )

        # Incremental capacity across all components
        original_weekly = sum(
            capacity
            for (supplier, component), capacity
            in base_supplier_capacity.items()
            if supplier == supplier_id
        )

        updated_weekly = sum(
            capacity
            for (supplier, component), capacity
            in modified_capacity.items()
            if supplier == supplier_id
        )

        extra_weekly = updated_weekly - original_weekly

        name = (
            f"Supplier_{supplier_id}_Plus_{increase_pct}Pct"
        )

        result = run_sensitivity_experiment(
            name,
            {
                "supplier_capacity": modified_capacity
            }
        )

        sensitivity_results[name] = result

        experiment_metadata.append({
            "Experiment": name,
            "Category": "Supplier Capacity",
            "Lever": supplier_id,
            "Change_Pct": increase_pct,
            "Extra_Weekly_Capacity": extra_weekly,
            "Additional_Customer_Demand_Units": 0
        })

        supplier_experiments.append({
            "Supplier": supplier_id,
            "Capacity_Increase_Pct": increase_pct,
            "Extra_Weekly_Capacity": extra_weekly,
            "Status": result["Status"],
            "Optimized_Cost_USD": (
                result.get("Total_Cost_USD", np.nan)
            ),
            "Savings_USD": (
                reference_cost -
                result.get("Total_Cost_USD", np.nan)
            )
        })

supplier_sensitivity = pd.DataFrame(
    supplier_experiments
)

# Expanding capacity must not worsen the optimum
assert (
    supplier_sensitivity["Status"] == "Optimal"
).all()

assert (
    supplier_sensitivity["Savings_USD"] >= -0.01
).all()

print("SUPPLIER CAPACITY SENSITIVITY")
display(supplier_sensitivity.round(2))

print("\nExperiments completed:", len(supplier_sensitivity))
print("STEP 5: PASSED")


SUPPLIER CAPACITY SENSITIVITY


,Supplier,Capacity_Increase_Pct,Extra_Weekly_Capacity,Status,Optimized_Cost_USD,Savings_USD
0,SUP_A,10,135,Optimal,399345.2,4786.0
1,SUP_A,20,270,Optimal,396620.2,7511.0
2,SUP_A,30,405,Optimal,396248.2,7883.0
3,SUP_B,10,75,Optimal,404131.2,0.0
4,SUP_B,20,150,Optimal,404131.2,0.0
5,SUP_B,30,225,Optimal,404131.2,0.0



Experiments completed: 6
STEP 5: PASSED


In [5]:
# STEP 6: Manufacturing capacity sensitivity

manufacturing_experiments = []

for increase_pct in [5, 10, 20]:

    factor = 1 + increase_pct / 100

    modified_capacity = int(
        np.floor(base_plant_capacity * factor)
    )

    extra_weekly = (
        modified_capacity - base_plant_capacity
    )

    name = f"Manufacturing_Plus_{increase_pct}Pct"

    result = run_sensitivity_experiment(
        name,
        {
            "plant_capacity": modified_capacity
        }
    )

    sensitivity_results[name] = result

    experiment_metadata.append({
        "Experiment": name,
        "Category": "Manufacturing Capacity",
        "Lever": "Assembly Plant",
        "Change_Pct": increase_pct,
        "Extra_Weekly_Capacity": extra_weekly,
        "Additional_Customer_Demand_Units": 0
    })

    manufacturing_experiments.append({
        "Experiment": name,
        "Capacity_Increase_Pct": increase_pct,
        "Original_Weekly_Capacity": base_plant_capacity,
        "New_Weekly_Capacity": modified_capacity,
        "Extra_Weekly_Capacity": extra_weekly,
        "Status": result["Status"],
        "Optimized_Cost_USD": result.get(
            "Total_Cost_USD", np.nan
        ),
        "Savings_USD": (
            reference_cost -
            result.get("Total_Cost_USD", np.nan)
        )
    })

manufacturing_sensitivity = pd.DataFrame(
    manufacturing_experiments
)

assert (
    manufacturing_sensitivity["Status"] == "Optimal"
).all()

assert (
    manufacturing_sensitivity["Savings_USD"] >= -0.01
).all()

print("MANUFACTURING CAPACITY SENSITIVITY")
display(manufacturing_sensitivity.round(2))

print("\nOriginal weekly capacity:", base_plant_capacity)
print("Experiments completed:", len(manufacturing_sensitivity))
print("STEP 6: PASSED")

MANUFACTURING CAPACITY SENSITIVITY


,Experiment,Capacity_Increase_Pct,Original_Weekly_Capacity,New_Weekly_Capacity,Extra_Weekly_Capacity,Status,Optimized_Cost_USD,Savings_USD
0,Manufacturing_Plus_5Pct,5,580,609,29,Optimal,403891.2,240.0
1,Manufacturing_Plus_10Pct,10,580,638,58,Optimal,403775.2,356.0
2,Manufacturing_Plus_20Pct,20,580,696,116,Optimal,403623.2,508.0



Original weekly capacity: 580
Experiments completed: 3
STEP 6: PASSED


In [6]:
# STEP 7: Transportation freight price sensitivity

freight_experiments = []

for change_pct in [-25, 25, 50, 100]:

    factor = 1 + change_pct / 100

    modified_cost = base_lane_cost.copy()

    # Modify standard transportation costs only
    for lane_id, mode in base_lane_modes.items():

        if mode == "Standard":
            modified_cost[lane_id] = (
                base_lane_cost[lane_id] * factor
            )

    direction = (
        "Minus" if change_pct < 0 else "Plus"
    )

    name = (
        f"Standard_Freight_{direction}_"
        f"{abs(change_pct)}Pct"
    )

    result = run_sensitivity_experiment(
        name,
        {
            "lane_cost": modified_cost
        }
    )

    sensitivity_results[name] = result

    experiment_metadata.append({
        "Experiment": name,
        "Category": "Freight Economics",
        "Lever": "Standard Freight",
        "Change_Pct": change_pct,
        "Extra_Weekly_Capacity": 0,
        "Additional_Customer_Demand_Units": 0
    })

    if result["Status"] == "Optimal":

        shipments = result["Details"]["Shipments"]

        standard_units = int(
            shipments.loc[
                shipments["Lane_ID"].map(
                    base_lane_modes
                ) == "Standard",
                "Ship_Units"
            ].sum()
        )

        expedited_units = int(
            shipments.loc[
                shipments["Lane_ID"].map(
                    base_lane_modes
                ) == "Expedited",
                "Ship_Units"
            ].sum()
        )

        # Reprice the reoptimized schedule using
        # the ORIGINAL freight rates.
        freight_at_original_rates = sum(
            row.Ship_Units * base_lane_cost[row.Lane_ID]
            for row in shipments.itertuples(index=False)
        )

        total_at_original_rates = (
            result["Total_Cost_USD"]
            - result["Freight_Cost_USD"]
            + freight_at_original_rates
        )

        price_effect = (
            result["Total_Cost_USD"]
            - total_at_original_rates
        )

        decision_effect = (
            reference_cost - total_at_original_rates
        )

        # Reconcile price and changed decision effects
        assert np.isclose(
            result["Total_Cost_USD"] - reference_cost,
            price_effect - decision_effect,
            atol=0.01
        )

    else:
        standard_units = np.nan
        expedited_units = np.nan
        price_effect = np.nan
        decision_effect = np.nan

    freight_experiments.append({
        "Experiment": name,
        "Standard_Freight_Change_Pct": change_pct,
        "Status": result["Status"],
        "Optimized_Cost_USD": result.get(
            "Total_Cost_USD", np.nan
        ),
        "Cost_Change_USD": (
            result.get("Total_Cost_USD", np.nan)
            - reference_cost
        ),
        "Standard_Shipment_Units": standard_units,
        "Expedited_Shipment_Units": expedited_units,
        "Freight_Price_Effect_USD": price_effect,
        "Reoptimization_Decision_Value_USD":
            decision_effect
    })

freight_sensitivity = pd.DataFrame(
    freight_experiments
)

assert (
    freight_sensitivity["Status"] == "Optimal"
).all()

print("FREIGHT COST SENSITIVITY")
display(freight_sensitivity.round(2))

print("\nExperiments completed:", len(freight_sensitivity))
print("STEP 7: PASSED")


FREIGHT COST SENSITIVITY


,Experiment,Standard_Freight_Change_Pct,Status,Optimized_Cost_USD,Cost_Change_USD,Standard_Shipment_Units,Expedited_Shipment_Units,Freight_Price_Effect_USD,Reoptimization_Decision_Value_USD
0,Standard_Freight_Minus_25Pct,-25,Optimal,398884.58,-5246.62,5103,562,-5246.62,0.0
1,Standard_Freight_Plus_25Pct,25,Optimal,409215.32,5084.12,4843,822,5019.12,-65.0
2,Standard_Freight_Plus_50Pct,50,Optimal,413920.45,9789.25,4128,1537,8705.25,-1084.0
3,Standard_Freight_Plus_100Pct,100,Optimal,421925.20,17794.00,3204,2461,11634.00,-6160.0



Experiments completed: 4
STEP 7: PASSED


In [7]:
# STEP 8: Demand growth sensitivity

demand_experiments = []

base_total_demand = sum(base_demand.values())

for growth_pct in [5, 10, 15, 20]:

    factor = 1 + growth_pct / 100

    modified_demand = {
        key: int(np.floor(value * factor + 0.5))
        for key, value in base_demand.items()
    }

    new_total_demand = sum(modified_demand.values())

    additional_units = (
        new_total_demand - base_total_demand
    )

    name = f"Demand_Growth_Plus_{growth_pct}Pct"

    result = run_sensitivity_experiment(
        name,
        {
            "base_demand": modified_demand
        }
    )

    sensitivity_results[name] = result

    experiment_metadata.append({
        "Experiment": name,
        "Category": "Demand Growth",
        "Lever": "All Regions",
        "Change_Pct": growth_pct,
        "Extra_Weekly_Capacity": 0,
        "Additional_Customer_Demand_Units":
            additional_units
    })

    status = result["Status"]

    if status == "Optimal":

        incremental_cost = (
            result["Total_Cost_USD"]
            - reference_cost
        )

        incremental_cost_per_unit = (
            incremental_cost / additional_units
        )

    else:
        incremental_cost = np.nan
        incremental_cost_per_unit = np.nan

    demand_experiments.append({
        "Experiment": name,
        "Demand_Growth_Pct": growth_pct,
        "Total_Demand_Units": new_total_demand,
        "Additional_Demand_Units": additional_units,
        "Status": status,
        "Optimized_Cost_USD": result.get(
            "Total_Cost_USD", np.nan
        ),
        "Incremental_Cost_USD": incremental_cost,
        "Incremental_Cost_Per_Added_Unit_USD":
            incremental_cost_per_unit
    })

demand_sensitivity = pd.DataFrame(
    demand_experiments
)

print("DEMAND GROWTH SENSITIVITY")
display(demand_sensitivity.round(2))

print("\nOriginal demand:", base_total_demand)
print("Experiments completed:", len(demand_sensitivity))

print("\nSTEP 8: PASSED")


DEMAND GROWTH SENSITIVITY


,Experiment,Demand_Growth_Pct,Total_Demand_Units,Additional_Demand_Units,Status,Optimized_Cost_USD,Incremental_Cost_USD,Incremental_Cost_Per_Added_Unit_USD
0,Demand_Growth_Plus_5Pct,5,6480,300,Optimal,433341.45,29210.25,97.37
1,Demand_Growth_Plus_10Pct,10,6804,624,Optimal,467764.55,63633.35,101.98
2,Demand_Growth_Plus_15Pct,15,7104,924,infeasible,NaN,NaN,NaN
3,Demand_Growth_Plus_20Pct,20,7416,1236,infeasible,NaN,NaN,NaN



Original demand: 6180
Experiments completed: 4

STEP 8: PASSED


In [8]:
# STEP 9A: Consolidate sensitivity experiment results

metadata = pd.DataFrame(experiment_metadata)

results_rows = []
cost_rows = []
operations_rows = []

cost_categories = {
    "Procurement": "Procurement_Cost_USD",
    "Manufacturing": "Manufacturing_Cost_USD",
    "Transportation": "Freight_Cost_USD",
    "DC Inventory Holding": "Holding_Cost_USD"
}

for row in metadata.itertuples(index=False):

    result = sensitivity_results[row.Experiment]
    optimal = result["Status"] == "Optimal"

    total_cost = (
        result["Total_Cost_USD"]
        if optimal else np.nan
    )

    results_rows.append({
        "Experiment": row.Experiment,
        "Category": row.Category,
        "Lever": row.Lever,
        "Change_Pct": row.Change_Pct,
        "Status": result["Status"],
        "Independent_Audit": result["Independent_Audit"],
        "Optimized_Cost_USD": total_cost,
        "Cost_Change_USD": (
            total_cost - reference_cost
            if optimal else np.nan
        ),
        "Cost_Change_Pct": (
            (total_cost - reference_cost)
            / reference_cost * 100
            if optimal else np.nan
        ),
        "Extra_Weekly_Capacity":
            row.Extra_Weekly_Capacity,
        "Additional_Customer_Demand_Units":
            row.Additional_Customer_Demand_Units
    })

    if not optimal:
        continue

    # Cost decomposition
    for category, field in cost_categories.items():

        cost_rows.append({
            "Experiment": row.Experiment,
            "Category": row.Category,
            "Cost_Category": category,
            "Cost_USD": result[field]
        })

    # Operational decisions
    details = result["Details"]
    orders = details["Purchases"]
    production = details["Production"]
    shipments = details["Shipments"]
    dc_inventory = details["DC_Inventory"]

    expedited_units = shipments.loc[
        shipments["Lane_ID"].map(
            base_lane_modes
        ) == "Expedited",
        "Ship_Units"
    ].sum()

    total_shipped = shipments["Ship_Units"].sum()

    operations_rows.append({
        "Experiment": row.Experiment,
        "Category": row.Category,
        "Supplier_A_Orders": orders.loc[
            orders["Supplier_ID"] == "SUP_A",
            "Order_Units"
        ].sum(),
        "Supplier_B_Orders": orders.loc[
            orders["Supplier_ID"] == "SUP_B",
            "Order_Units"
        ].sum(),
        "Total_Production_Units":
            production["Production_Units"].sum(),
        "Peak_Weekly_Production":
            production["Production_Units"].max(),
        "Total_Shipped_Units": total_shipped,
        "Expedited_Shipment_Units": expedited_units,
        "Expedited_Share_Pct": (
            expedited_units / total_shipped * 100
        ),
        "Average_DC_Ending_Inventory":
            dc_inventory.groupby("Week")[
                "Ending_Inventory"
            ].sum().mean()
    })

sensitivity_comparison = pd.DataFrame(
    results_rows
)

sensitivity_cost_breakdown = pd.DataFrame(
    cost_rows
)

sensitivity_operations = pd.DataFrame(
    operations_rows
)

# --------------------------------------------------
# Validate experiment completeness
# --------------------------------------------------

assert len(sensitivity_comparison) == 18
assert sensitivity_comparison["Experiment"].is_unique

assert len(sensitivity_results) == 18

assert (
    sensitivity_comparison.loc[
        sensitivity_comparison["Status"] == "Optimal",
        "Independent_Audit"
    ] == "PASSED"
).all()

# Reconcile component costs to optimized total
cost_reconciliation = (
    sensitivity_cost_breakdown.groupby(
        "Experiment"
    )["Cost_USD"].sum()
)

for name, calculated_cost in cost_reconciliation.items():
    assert np.isclose(
        calculated_cost,
        sensitivity_results[name]["Total_Cost_USD"],
        atol=0.01
    )

print("CONSOLIDATED SENSITIVITY RESULTS")
display(sensitivity_comparison.round(2))

print("\nExperiments:", len(sensitivity_comparison))
print("Optimal:", (
    sensitivity_comparison["Status"] == "Optimal"
).sum())

print("Non-optimal:", (
    sensitivity_comparison["Status"] != "Optimal"
).sum())

print("\nCOST RECONCILIATION: PASSED")
print("STEP 9A: PASSED")


CONSOLIDATED SENSITIVITY RESULTS


,Experiment,Category,Lever,Change_Pct,Status,Independent_Audit,Optimized_Cost_USD,Cost_Change_USD,Cost_Change_Pct,Extra_Weekly_Capacity,Additional_Customer_Demand_Units
0,Reference,Reference,Original Network,0,Optimal,PASSED,404131.20,0.00,0.00,0,0
1,Supplier_SUP_A_Plus_10Pct,Supplier Capacity,SUP_A,10,Optimal,PASSED,399345.20,-4786.00,-1.18,135,0
2,Supplier_SUP_A_Plus_20Pct,Supplier Capacity,SUP_A,20,Optimal,PASSED,396620.20,-7511.00,-1.86,270,0
3,Supplier_SUP_A_Plus_30Pct,Supplier Capacity,SUP_A,30,Optimal,PASSED,396248.20,-7883.00,-1.95,405,0
4,Supplier_SUP_B_Plus_10Pct,Supplier Capacity,SUP_B,10,Optimal,PASSED,404131.20,0.00,0.00,75,0
5,Supplier_SUP_B_Plus_20Pct,Supplier Capacity,SUP_B,20,Optimal,PASSED,404131.20,0.00,0.00,150,0
6,Supplier_SUP_B_Plus_30Pct,Supplier Capacity,SUP_B,30,Optimal,PASSED,404131.20,0.00,0.00,225,0
7,Manufacturing_Plus_5Pct,Manufacturing Capacity,Assembly Plant,5,Optimal,PASSED,403891.20,-240.00,-0.06,29,0
8,Manufacturing_Plus_10Pct,Manufacturing Capacity,Assembly Plant,10,Optimal,PASSED,403775.20,-356.00,-0.09,58,0
9,Manufacturing_Plus_20Pct,Manufacturing Capacity,Assembly Plant,20,Optimal,PASSED,403623.20,-508.00,-0.13,116,0



Experiments: 18
Optimal: 16
Non-optimal: 2

COST RECONCILIATION: PASSED
STEP 9A: PASSED


In [9]:
# STEP 9B: Calculate capacity investment value

capacity_results = sensitivity_comparison.loc[
    sensitivity_comparison["Category"].isin([
        "Supplier Capacity",
        "Manufacturing Capacity"
    ])
].copy()

assert (
    capacity_results["Status"] == "Optimal"
).all()

capacity_value_rows = []

for (category, lever), group in capacity_results.groupby(
    ["Category", "Lever"]
):

    group = group.sort_values(
        "Extra_Weekly_Capacity"
    )

    previous_cost = reference_cost
    previous_extra_capacity = 0

    for row in group.itertuples(index=False):

        # Total benefit versus original reference
        total_savings = (
            reference_cost - row.Optimized_Cost_USD
        )

        extra_capacity_unit_weeks = (
            row.Extra_Weekly_Capacity
            * planning_horizon
        )

        # Incremental benefit versus previous expansion
        interval_savings = (
            previous_cost - row.Optimized_Cost_USD
        )

        interval_extra_capacity = (
            row.Extra_Weekly_Capacity
            - previous_extra_capacity
        )

        interval_capacity_unit_weeks = (
            interval_extra_capacity * planning_horizon
        )

        capacity_value_rows.append({
            "Category": category,
            "Lever": lever,
            "Change_Pct": row.Change_Pct,
            "Extra_Weekly_Capacity":
                row.Extra_Weekly_Capacity,
            "Extra_Capacity_Unit_Weeks":
                extra_capacity_unit_weeks,
            "Total_Savings_USD": total_savings,
            "Maximum_12_Week_Investment_USD":
                max(0, total_savings),
            "Break_Even_Premium_Per_Unit_Week_USD":
                (
                    total_savings / extra_capacity_unit_weeks
                    if extra_capacity_unit_weeks > 0
                    else np.nan
                ),
            "Interval_Savings_USD": interval_savings,
            "Interval_Capacity_Unit_Weeks":
                interval_capacity_unit_weeks,
            "Marginal_Value_Per_Unit_Week_USD":
                (
                    interval_savings
                    / interval_capacity_unit_weeks
                    if interval_capacity_unit_weeks > 0
                    else np.nan
                )
        })

        previous_cost = row.Optimized_Cost_USD
        previous_extra_capacity = (
            row.Extra_Weekly_Capacity
        )

capacity_value = pd.DataFrame(
    capacity_value_rows
)

assert len(capacity_value) == 9

assert (
    capacity_value["Total_Savings_USD"] >= -0.01
).all()

print("CAPACITY INVESTMENT VALUE")
display(capacity_value.round(4))

print("\nSTEP 9B: PASSED")


CAPACITY INVESTMENT VALUE


,Category,Lever,Change_Pct,Extra_Weekly_Capacity,Extra_Capacity_Unit_Weeks,Total_Savings_USD,Maximum_12_Week_Investment_USD,Break_Even_Premium_Per_Unit_Week_USD,Interval_Savings_USD,Interval_Capacity_Unit_Weeks,Marginal_Value_Per_Unit_Week_USD
0,Manufacturing Capacity,Assembly Plant,5,29,348,240.0,240.0,0.6897,240.0,348,0.6897
1,Manufacturing Capacity,Assembly Plant,10,58,696,356.0,356.0,0.5115,116.0,348,0.3333
2,Manufacturing Capacity,Assembly Plant,20,116,1392,508.0,508.0,0.3649,152.0,696,0.2184
3,Supplier Capacity,SUP_A,10,135,1620,4786.0,4786.0,2.9543,4786.0,1620,2.9543
4,Supplier Capacity,SUP_A,20,270,3240,7511.0,7511.0,2.3182,2725.0,1620,1.6821
5,Supplier Capacity,SUP_A,30,405,4860,7883.0,7883.0,1.6220,372.0,1620,0.2296
6,Supplier Capacity,SUP_B,10,75,900,0.0,0.0,0.0000,0.0,900,0.0000
7,Supplier Capacity,SUP_B,20,150,1800,0.0,0.0,0.0000,0.0,900,0.0000
8,Supplier Capacity,SUP_B,30,225,2700,0.0,0.0,0.0000,0.0,900,0.0000



STEP 9B: PASSED


In [10]:
# STEP 9C: Identify the most valuable capacity levers

capacity_recommendations = (
    capacity_value.sort_values(
        [
            "Total_Savings_USD",
            "Break_Even_Premium_Per_Unit_Week_USD"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

capacity_recommendations["Economic_Interpretation"] = (
    np.where(
        capacity_recommendations["Total_Savings_USD"] > 0.01,
        "Positive modeled value",
        "No demonstrated savings"
    )
)

print("CAPACITY DECISION OPPORTUNITIES")
display(
    capacity_recommendations[[
        "Category",
        "Lever",
        "Change_Pct",
        "Extra_Weekly_Capacity",
        "Total_Savings_USD",
        "Break_Even_Premium_Per_Unit_Week_USD",
        "Economic_Interpretation"
    ]].round(4)
)

print("\nSTEP 9C: PASSED")


CAPACITY DECISION OPPORTUNITIES


,Category,Lever,Change_Pct,Extra_Weekly_Capacity,Total_Savings_USD,Break_Even_Premium_Per_Unit_Week_USD,Economic_Interpretation
0,Supplier Capacity,SUP_A,30,405,7883.0,1.6220,Positive modeled value
1,Supplier Capacity,SUP_A,20,270,7511.0,2.3182,Positive modeled value
2,Supplier Capacity,SUP_A,10,135,4786.0,2.9543,Positive modeled value
3,Manufacturing Capacity,Assembly Plant,20,116,508.0,0.3649,Positive modeled value
4,Manufacturing Capacity,Assembly Plant,10,58,356.0,0.5115,Positive modeled value
5,Manufacturing Capacity,Assembly Plant,5,29,240.0,0.6897,Positive modeled value
6,Supplier Capacity,SUP_B,10,75,0.0,0.0000,No demonstrated savings
7,Supplier Capacity,SUP_B,20,150,0.0,0.0000,No demonstrated savings
8,Supplier Capacity,SUP_B,30,225,0.0,0.0000,No demonstrated savings



STEP 9C: PASSED


In [11]:
# STEP 10A: Export Tableau-ready sensitivity datasets

output_dir = repo_root / "outputs" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

# Separate audit-status table
sensitivity_audit_summary = sensitivity_comparison[[
    "Experiment",
    "Category",
    "Status",
    "Independent_Audit"
]].copy()

# Executive KPI reporting table
sensitivity_dashboard_kpis = (
    sensitivity_comparison[[
        "Experiment",
        "Category",
        "Lever",
        "Change_Pct",
        "Status",
        "Optimized_Cost_USD",
        "Cost_Change_USD",
        "Cost_Change_Pct"
    ]]
    .copy()
)

sensitivity_dashboard_kpis["Reference_Cost_USD"] = (
    reference_cost
)

sensitivity_dashboard_kpis["Savings_USD"] = (
    reference_cost
    - sensitivity_dashboard_kpis["Optimized_Cost_USD"]
)

# Export mapping
exports = {
    "sensitivity_comparison.csv":
        sensitivity_comparison,

    "sensitivity_supplier_capacity.csv":
        supplier_sensitivity,

    "sensitivity_manufacturing_capacity.csv":
        manufacturing_sensitivity,

    "sensitivity_freight_economics.csv":
        freight_sensitivity,

    "sensitivity_demand_growth.csv":
        demand_sensitivity,

    "sensitivity_capacity_investment_value.csv":
        capacity_value,

    "sensitivity_capacity_recommendations.csv":
        capacity_recommendations,

    "sensitivity_cost_breakdown.csv":
        sensitivity_cost_breakdown,

    "sensitivity_operational_kpis.csv":
        sensitivity_operations,

    "sensitivity_audit_summary.csv":
        sensitivity_audit_summary,

    "sensitivity_dashboard_kpis.csv":
        sensitivity_dashboard_kpis
}

for filename, dataframe in exports.items():

    assert len(dataframe) > 0, (
        f"Empty output: {filename}"
    )

    output_path = output_dir / filename

    dataframe.to_csv(
        output_path,
        index=False
    )

    # Verify each exported file can be read
    reloaded = pd.read_csv(output_path)

    assert len(reloaded) == len(dataframe)

    print("Exported:", filename)

print("\nSENSITIVITY REPORTING EXPORT COMPLETE")
print("=" * 50)
print("Files exported:", len(exports))
print("Output directory:", output_dir)
print("STEP 10A: PASSED")


Exported: sensitivity_comparison.csv
Exported: sensitivity_supplier_capacity.csv
Exported: sensitivity_manufacturing_capacity.csv
Exported: sensitivity_freight_economics.csv
Exported: sensitivity_demand_growth.csv
Exported: sensitivity_capacity_investment_value.csv
Exported: sensitivity_capacity_recommendations.csv
Exported: sensitivity_cost_breakdown.csv
Exported: sensitivity_operational_kpis.csv
Exported: sensitivity_audit_summary.csv
Exported: sensitivity_dashboard_kpis.csv

SENSITIVITY REPORTING EXPORT COMPLETE
Files exported: 11
Output directory: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/outputs/tables
STEP 10A: PASSED


In [12]:
# STEP 11A: Generate sensitivity analysis executive summary

from IPython.display import Markdown, display

# --------------------------------------------------
# Identify capacity opportunities
# --------------------------------------------------

positive_capacity = capacity_recommendations.loc[
    capacity_recommendations["Total_Savings_USD"] > 0.01
]

if len(positive_capacity) > 0:

    best = positive_capacity.iloc[0]

    capacity_finding = (
        f"The highest absolute modeled savings among "
        f"tested capacity expansions came from "
        f"**{best['Lever']} +{best['Change_Pct']:.0f}%**, "
        f"generating **${best['Total_Savings_USD']:,.2f}** "
        f"over 12 weeks before expansion costs."
    )

else:
    capacity_finding = (
        "None of the tested capacity expansions reduced "
        "the original optimized network cost. "
        "Capacity expansion is therefore not supported "
        "by base-case modeled cost savings alone."
    )

# --------------------------------------------------
# Summarize demand growth feasibility
# --------------------------------------------------

feasible_demand = demand_sensitivity.loc[
    demand_sensitivity["Status"] == "Optimal"
]

unverified_demand = demand_sensitivity.loc[
    demand_sensitivity["Status"] != "Optimal"
]

if len(feasible_demand) > 0:

    max_feasible_growth = (
        feasible_demand["Demand_Growth_Pct"].max()
    )

    demand_finding = (
        f"The largest tested demand increase with a "
        f"verified optimal full-service solution was "
        f"**+{max_feasible_growth:.0f}%**."
    )

else:
    demand_finding = (
        "No positive demand growth experiment produced "
        "a verified optimal full-service solution."
    )

if len(unverified_demand) > 0:

    demand_finding += (
        " Other tested growth levels did not produce "
        "verified optimal solutions and require "
        "status-specific interpretation."
    )

# --------------------------------------------------
# Summarize freight sensitivity
# --------------------------------------------------

highest_freight = freight_sensitivity.sort_values(
    "Standard_Freight_Change_Pct"
).iloc[-1]

freight_finding = (
    f"At the highest tested standard freight increase "
    f"(+{highest_freight['Standard_Freight_Change_Pct']:.0f}%), "
    f"the reoptimized network cost was "
    f"**${highest_freight['Optimized_Cost_USD']:,.2f}**, "
    f"compared with the original "
    f"**${reference_cost:,.2f}**."
)

# --------------------------------------------------
# Format complete summary
# --------------------------------------------------

summary = f"""
# Notebook 08 — Sensitivity & Decision Analysis

## Executive Summary

This analysis evaluates the sensitivity of a
hypothetical consumer electronics supply network
to capacity, freight, and demand assumptions.

All scenarios were modeled using the same 12-week
Pyomo optimization framework validated in
Notebooks 04 and 07.

## Reference Network

- Optimized cost: **${reference_cost:,.2f}**
- Planning horizon: **12 weeks**
- Original demand: **{base_total_demand:,} units**
- Customer fulfillment requirement: **100%**
- Original terminal inventory requirements: **Preserved**

## Experiment Coverage

- Supplier capacity experiments: **6**
- Manufacturing capacity experiments: **3**
- Standard freight price experiments: **4**
- Demand growth experiments: **4**
- Reference case: **1**
- Total optimization experiments: **18**

## Findings

### 1. Supplier and Manufacturing Capacity

{capacity_finding}

Capacity savings represent an upper bound on
justifiable expansion spending for the modeled
12-week cost benefit, before accounting for
unmodeled operational and risk considerations.

### 2. Transportation Economics

{freight_finding}

Freight sensitivity includes both the direct
effect of modified transportation prices
and the network's operational response.

### 3. Demand Growth

{demand_finding}

Infeasible or otherwise non-optimal outcomes
must be distinguished from verified optimal
full-service solutions.

### 4. Investment Decision Framework

For each capacity experiment, the analysis measures:

- Total savings against the original network
- Extra weekly capacity
- Additional capacity unit-weeks
- Break-even capacity premium
- Marginal value of each expansion interval

Capacity values are specific to the tested
network and planning horizon.

## Validation

- Reference model cost reconciled
- Original parameters restored after experiments
- Every optimal result independently audited
- Cost-category totals reconciled
- Tableau reporting datasets exported

## Limitations

- Hypothetical network parameters
- Deterministic 12-week optimization
- Fixed supplier and transportation lead times
- No capital expenditure estimates
- No explicit supplier reservation fees
- No probabilistic demand uncertainty
- No quantified disruption risk reduction
- Original terminal inventories preserved

## Business Recommendations

1. Prioritize capacity investments only where
   demonstrable operational value exceeds the
   full cost of implementation.
2. Use sensitivity experiments to distinguish
   genuinely binding constraints from
   apparently constrained but low-value resources.
3. Evaluate transportation decisions jointly
   with inventory and production planning.
4. Use demand-growth feasibility tests to
   identify conditions requiring capacity or
   network redesign.
5. Extend the analysis with resilience
   benefits before making final investments.

## Next Steps

1. Build executive Tableau dashboards.
2. Integrate scenario and sensitivity reporting.
3. Perform end-to-end reproducibility checks.
4. Finalize the Project 2 portfolio case study.
"""

# Display in Jupyter notebook
display(Markdown(summary))

# Save final findings
docs_dir = repo_root / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

summary_file = docs_dir / "notebook08_findings.md"

summary_file.write_text(
    summary,
    encoding="utf-8"
)

assert summary_file.is_file()

print("Executive summary saved:", summary_file)
print("STEP 11A: PASSED")


# Notebook 08 — Sensitivity & Decision Analysis

## Executive Summary

This analysis evaluates the sensitivity of a
hypothetical consumer electronics supply network
to capacity, freight, and demand assumptions.

All scenarios were modeled using the same 12-week
Pyomo optimization framework validated in
Notebooks 04 and 07.

## Reference Network

- Optimized cost: **$404,131.20**
- Planning horizon: **12 weeks**
- Original demand: **6,180 units**
- Customer fulfillment requirement: **100%**
- Original terminal inventory requirements: **Preserved**

## Experiment Coverage

- Supplier capacity experiments: **6**
- Manufacturing capacity experiments: **3**
- Standard freight price experiments: **4**
- Demand growth experiments: **4**
- Reference case: **1**
- Total optimization experiments: **18**

## Findings

### 1. Supplier and Manufacturing Capacity

The highest absolute modeled savings among tested capacity expansions came from **SUP_A +30%**, generating **$7,883.00** over 12 weeks before expansion costs.

Capacity savings represent an upper bound on
justifiable expansion spending for the modeled
12-week cost benefit, before accounting for
unmodeled operational and risk considerations.

### 2. Transportation Economics

At the highest tested standard freight increase (+100%), the reoptimized network cost was **$421,925.20**, compared with the original **$404,131.20**.

Freight sensitivity includes both the direct
effect of modified transportation prices
and the network's operational response.

### 3. Demand Growth

The largest tested demand increase with a verified optimal full-service solution was **+10%**. Other tested growth levels did not produce verified optimal solutions and require status-specific interpretation.

Infeasible or otherwise non-optimal outcomes
must be distinguished from verified optimal
full-service solutions.

### 4. Investment Decision Framework

For each capacity experiment, the analysis measures:

- Total savings against the original network
- Extra weekly capacity
- Additional capacity unit-weeks
- Break-even capacity premium
- Marginal value of each expansion interval

Capacity values are specific to the tested
network and planning horizon.

## Validation

- Reference model cost reconciled
- Original parameters restored after experiments
- Every optimal result independently audited
- Cost-category totals reconciled
- Tableau reporting datasets exported

## Limitations

- Hypothetical network parameters
- Deterministic 12-week optimization
- Fixed supplier and transportation lead times
- No capital expenditure estimates
- No explicit supplier reservation fees
- No probabilistic demand uncertainty
- No quantified disruption risk reduction
- Original terminal inventories preserved

## Business Recommendations

1. Prioritize capacity investments only where
   demonstrable operational value exceeds the
   full cost of implementation.
2. Use sensitivity experiments to distinguish
   genuinely binding constraints from
   apparently constrained but low-value resources.
3. Evaluate transportation decisions jointly
   with inventory and production planning.
4. Use demand-growth feasibility tests to
   identify conditions requiring capacity or
   network redesign.
5. Extend the analysis with resilience
   benefits before making final investments.

## Next Steps

1. Build executive Tableau dashboards.
2. Integrate scenario and sensitivity reporting.
3. Perform end-to-end reproducibility checks.
4. Finalize the Project 2 portfolio case study.


Executive summary saved: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/docs/notebook08_findings.md
STEP 11A: PASSED


In [13]:
# STEP 11B: Final Notebook 08 quality assurance

print("NOTEBOOK 08 — FINAL QUALITY ASSURANCE")
print("=" * 50)

# Expected experiment counts
assert len(supplier_sensitivity) == 6
assert len(manufacturing_sensitivity) == 3
assert len(freight_sensitivity) == 4
assert len(demand_sensitivity) == 4

assert len(sensitivity_comparison) == 18
print("Experiment coverage: PASSED")

# Reference validation
assert np.isclose(
    sensitivity_reference["Total_Cost_USD"],
    404131.20,
    atol=0.01
)
print("Reference reconciliation: PASSED")

# All optimal experiments audited
optimal_results = sensitivity_comparison.loc[
    sensitivity_comparison["Status"] == "Optimal"
]

assert (
    optimal_results["Independent_Audit"] == "PASSED"
).all()

print("Independent validation: PASSED")

# Ensure no parameters were accidentally changed
assert engine["supplier_capacity"] == base_supplier_capacity
assert engine["plant_capacity"] == base_plant_capacity
assert engine["lane_cost"] == base_lane_cost
assert engine["base_demand"] == base_demand

print("Parameter restoration: PASSED")

# Verify reporting files
for filename in exports:
    assert (output_dir / filename).is_file()

assert summary_file.is_file()

print("Reporting files: PASSED")
print("Executive summary: PASSED")

# Report non-optimal cases transparently
nonoptimal = sensitivity_comparison.loc[
    sensitivity_comparison["Status"] != "Optimal"
]

if len(nonoptimal) > 0:

    print("\nNON-OPTIMAL EXPERIMENTS:")
    display(
        nonoptimal[[
            "Experiment", "Status"
        ]]
    )

    print(
        "Review solver statuses. Non-optimal does "
        "not automatically mean proven infeasible."
    )

print("\n" + "=" * 50)
print("NOTEBOOK 08: COMPLETE")
print("=" * 50)
print("Total experiments:", len(sensitivity_comparison))
print("Verified optimal experiments:", len(optimal_results))
print("Reporting CSVs:", len(exports))
print("Executive summary:", summary_file.name)


NOTEBOOK 08 — FINAL QUALITY ASSURANCE
Experiment coverage: PASSED
Reference reconciliation: PASSED
Independent validation: PASSED
Parameter restoration: PASSED
Reporting files: PASSED
Executive summary: PASSED

NON-OPTIMAL EXPERIMENTS:


,Experiment,Status
16,Demand_Growth_Plus_15Pct,infeasible
17,Demand_Growth_Plus_20Pct,infeasible


Review solver statuses. Non-optimal does not automatically mean proven infeasible.

NOTEBOOK 08: COMPLETE
Total experiments: 18
Verified optimal experiments: 16
Reporting CSVs: 11
Executive summary: notebook08_findings.md
